# Jupyter Notebook to analyze human-to-human attention

## Imports and Loading Data

In [ ]:
import sys
import os
import pickle
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.stats import pearsonr
import copy


In [ ]:
#%% Load all the analyzed data
def load_and_combine_pickles(directory, prefix):
    """
    Loads multiple pickle files containing dictionaries and combines them into a single dictionary.
    Allows for specification of the filename prefix for flexibility in file selection.

    Args:
    directory (str): The directory containing the pickle files.
    prefix (str): The prefix that the filenames of the pickle files should start with.

    Returns:
    dict: A combined dictionary containing keys and values from all the loaded dictionaries.
    """
    combined_dict = {}

    # List all files in the given directory
    for filename in os.listdir(directory):
        if filename.startswith(prefix) and filename.endswith(".pkl"):
            filepath = os.path.join(directory, filename)
            with open(filepath, 'rb') as file:
                # Load the dictionary from the pickle file
                data = pickle.load(file)
                if isinstance(data, dict):
                    # Combine the loaded dictionary into the master dictionary
                    combined_dict.update(data)

    return combined_dict

path_output = './output/r3_analysis_results/normalize_saliency_maps_to_sum_to_1/'
intra_prefix = "intra_question_analysis_results_videos_"
intra_random_control_prefix = "intra_question_analysis_random_control_results_videos_"

intra_all = load_and_combine_pickles(path_output, intra_prefix)
intra_ctr = load_and_combine_pickles(path_output, intra_random_control_prefix)
print(intra_all.keys())
print(intra_ctr.keys())

In [ ]:
load_and_process_inter_data = True # inter_question data is large, overloading RAM. Turn off if not necessary.
inter_prefix = "inter_question_analysis_results_videos_"
inter_all = load_and_combine_pickles(path_output, inter_prefix)
print(inter_all.keys())

## Analysis 1: using averaging across frames

### based on Normalized Saliency Scores (NSS)

In [ ]:
#%% Normalized Saliency Score (NSS) (intra question)
intra_nss_averaged_across_frames = []
for vq_no, vq_data in intra_all.items():
    if vq_no[0] >= 0:
        for user_id, user_saliency_data in vq_data.items():
            intra_nss_averaged_across_frames.append(sum(user_saliency_data['per_frame_nsscores'])/len(user_saliency_data['per_frame_nsscores']))
print(min(intra_nss_averaged_across_frames))
print(max(intra_nss_averaged_across_frames))

In [ ]:
plt.rcParams["font.family"] = "Times New Roman"
plt.figure(figsize=(3.3, 1.7))
fontsize = 9
bins = np.linspace(-1, 9, 28)
plt.hist(intra_nss_averaged_across_frames, bins=bins, label=f'NSS values \n(intra question)', color='r', density=True, alpha=0.5, hatch='///')
intra_nss_mean_value = sum(intra_nss_averaged_across_frames)/len(intra_nss_averaged_across_frames)
plt.axvline(intra_nss_mean_value, color='r', label=f'NSS mean={intra_nss_mean_value:.2f}\n(intra question)')
plt.title("NSS Histogram with Density", fontsize=fontsize)
plt.xlabel("Normalized Saliency Score (NSS)", fontsize=fontsize)
plt.ylabel("Probability Density Function", fontsize=fontsize)
plt.xticks([0, 2, 4, 6, 8], fontsize=fontsize)
plt.yticks([0.0, 0.1, 0.2, 0.3], fontsize=fontsize)
plt.tick_params(axis='both', direction='in', which='both', length=5, width=1)
plt.grid(True, which='major', linestyle='--', alpha=0.45)

plt.legend(loc="center right", ncol= 1, fontsize=fontsize, bbox_to_anchor=(1.02, 0.50))
plt.savefig('./output/r3_analysis_results/'+'NSS_averaged_across_frames_intra_question_resized.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from scipy.stats import sem, tstd

plt.rcParams["font.family"] = "Times New Roman"
plt.figure(figsize=(6.6, 3.5))
fontsize = 9
bins = np.linspace(-1, 9, 28)
plt.hist(intra_nss_averaged_across_frames, bins=bins, label=f'NSS values \n(intra question)', color='r', density=True, alpha=0.5, hatch='///')
intra_nss_mean_value = sum(intra_nss_averaged_across_frames)/len(intra_nss_averaged_across_frames)
intra_nss_stddev_val = tstd(intra_nss_averaged_across_frames)
plt.axvline(intra_nss_mean_value, color='r', label=f'NSS mean = {intra_nss_mean_value:.2f}\u00B1{intra_nss_stddev_val:.2f}\n(intra question)')
plt.title("NSS Histogram with Density", fontsize=fontsize)
plt.xlabel("Normalized Saliency Score (NSS)", fontsize=fontsize)
plt.ylabel("Probability Density Function", fontsize=fontsize)
plt.xticks([0, 2, 4, 6, 8], fontsize=fontsize)
plt.yticks([0.0, 0.1, 0.2, 0.3], fontsize=fontsize)
plt.tick_params(axis='both', direction='in', which='both', length=5, width=1)
plt.grid(True, which='major', linestyle='--', alpha=0.45)

plt.legend(loc="upper right", ncol= 1, fontsize=fontsize)#, bbox_to_anchor=(1.02, 0.50))
plt.savefig('./output/r3_analysis_results/'+'NSS_averaged_across_frames_intra_question_highres.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#%% Normalized Saliency Score (NSS) (inter question)
if load_and_process_inter_data:
    inter_nss_averaged_across_frames = []
    for vq_no, vq_data in inter_all.items():
        if vq_no[0] >= 0:
            for user_id, user_saliency_data in vq_data.items():
                inter_nss_averaged_across_frames.append(sum(user_saliency_data['per_frame_nsscores'])/len(user_saliency_data['per_frame_nsscores']))
print(min(inter_nss_averaged_across_frames))
print(max(inter_nss_averaged_across_frames))

In [ ]:
if load_and_process_inter_data:
    plt.rcParams["font.family"] = "Times New Roman"
    plt.figure(figsize=(3.3, 1.7))
    fontsize = 9
    bins = np.linspace(-1, 9, 28)
    plt.hist(inter_nss_averaged_across_frames, bins=bins, label=f'NSS values\n(inter question)', color='b', density=True, alpha=0.5)
    inter_nss_mean_value = sum(inter_nss_averaged_across_frames)/len(inter_nss_averaged_across_frames)
    plt.axvline(inter_nss_mean_value, color='b', label=f'NSS mean={inter_nss_mean_value:.2f}\n(inter question)')
    plt.hist(intra_nss_averaged_across_frames, bins=bins, label=f'NSS values\n(intra question)', color='r', density=True, alpha=0.5, hatch='///')
    intra_nss_mean_value = sum(intra_nss_averaged_across_frames)/len(intra_nss_averaged_across_frames)
    plt.axvline(intra_nss_mean_value, color='r', label=f'NSS mean={intra_nss_mean_value:.2f}\n(intra question)')


    plt.title("NSS Histogram with Density", fontsize=fontsize)
    plt.xlabel("Normalized Saliency Score (NSS)", fontsize=fontsize)
    plt.ylabel("Probability Density Function", fontsize=fontsize)
    plt.xticks(fontsize=fontsize)
    plt.yticks(fontsize=fontsize)
    plt.tick_params(axis='both', direction='in', which='both', length=5, width=1)
    plt.grid(True, which='major', linestyle='--', alpha=0.45)
    handles, labels = plt.gca().get_legend_handles_labels()

    # Change legend order by reordering handles and labels
    order = [2, 3, 0, 1]  # Example order: specify indices of desired order
    plt.legend(
        [handles[i] for i in order],
        [labels[i] for i in order],
        loc="center right",
        ncol=1,
        fontsize=7,
        bbox_to_anchor=(1.015, 0.60)
    )

    # plt.legend(loc="center right", ncol= 1, fontsize=fontsize, bbox_to_anchor=(1.02, 0.50))
    plt.savefig('./output/r3_analysis_results/'+'NSS_averaged_across_frames_intra_and_inter_question_resized.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
from scipy.stats import sem, tstd

if load_and_process_inter_data:
    plt.rcParams["font.family"] = "Times New Roman"
    plt.figure(figsize=(6.6, 3.5))
    fontsize = 9
    bins = np.linspace(-1, 9, 28)
    plt.hist(inter_nss_averaged_across_frames, bins=bins, label=f'NSS values\n(inter question)', color='b', density=True, alpha=0.5)
    inter_nss_mean_value = sum(inter_nss_averaged_across_frames)/len(inter_nss_averaged_across_frames)
    inter_nss_stddev_val = tstd(inter_nss_averaged_across_frames)
    plt.axvline(inter_nss_mean_value, color='b', label=f'NSS mean = {inter_nss_mean_value:.2f}\u00B1{inter_nss_stddev_val:.2f}\n(inter question)')
    plt.hist(intra_nss_averaged_across_frames, bins=bins, label=f'NSS values\n(intra question)', color='r', density=True, alpha=0.5, hatch='///')
    intra_nss_mean_value = sum(intra_nss_averaged_across_frames)/len(intra_nss_averaged_across_frames)
    intra_nss_stddev_val = tstd(intra_nss_averaged_across_frames)
    plt.axvline(intra_nss_mean_value, color='r', label=f'NSS mean = {intra_nss_mean_value:.2f}\u00B1{intra_nss_stddev_val:.2f}\n(intra question)')


    plt.title("NSS Histogram with Density", fontsize=fontsize)
    plt.xlabel("Normalized Saliency Score (NSS)", fontsize=fontsize)
    plt.ylabel("Probability Density Function", fontsize=fontsize)
    plt.xticks(fontsize=fontsize)
    plt.yticks(fontsize=fontsize)
    plt.tick_params(axis='both', direction='in', which='both', length=5, width=1)
    plt.grid(True, which='major', linestyle='--', alpha=0.45)
    handles, labels = plt.gca().get_legend_handles_labels()

    # Change legend order by reordering handles and labels
    order = [2, 3, 0, 1]  # Example order: specify indices of desired order
    plt.legend(
        [handles[i] for i in order],
        [labels[i] for i in order],
        loc="upper right",
        ncol=1,
        fontsize=fontsize
    )

    # plt.legend(loc="center right", ncol= 1, fontsize=fontsize, bbox_to_anchor=(1.02, 0.50))
    plt.savefig('./output/r3_analysis_results/'+'NSS_averaged_across_frames_intra_and_inter_question_highres.png', dpi=300, bbox_inches='tight')
    plt.show()

### based on Percentile (not used anymore)

In [ ]:
#%% Percentile 
intra_percentile_averaged_across_frames = []
for vq_no, vq_data in intra_all.items():
    if vq_no[0] >= 0:
        for user_id, user_saliency_data in vq_data.items():
            intra_percentile_averaged_across_frames.append(sum(user_saliency_data['per_frame_percentiles'])/len(user_saliency_data['per_frame_percentiles']))


inter_percentile_averaged_across_frames = []
for vq_no, vq_data in inter_all.items():
    if vq_no[0] >= 0:
        for user_id, user_saliency_data in vq_data.items():
            inter_percentile_averaged_across_frames.append(sum(user_saliency_data['per_frame_percentiles'])/len(user_saliency_data['per_frame_percentiles']))

bins = np.linspace(0, 100, 25)

plt.hist(intra_percentile_averaged_across_frames, bins=bins, label='intra_percentile_averaged_across_frames', color='r', density=True, alpha=0.5)
intra_percentile_mean_value = sum(intra_percentile_averaged_across_frames)/len(intra_percentile_averaged_across_frames)
plt.axvline(intra_percentile_mean_value, color='r', label='intra_percentile_mean={:.2f}'.format(intra_percentile_mean_value))

plt.hist(inter_percentile_averaged_across_frames, bins=bins, label='inter_percentile_averaged_across_frames', color='b', density=True, alpha=0.5)
inter_percentile_mean_value = sum(inter_percentile_averaged_across_frames)/len(inter_percentile_averaged_across_frames)
plt.axvline(inter_percentile_mean_value, color='b', label='inter_percentile_mean={:.2f}'.format(inter_percentile_mean_value))

plt.legend(loc="upper left")
plt.show()

## Analysis 2: using per frame averaging across participants

### initial verification codes

In [ ]:
#%% Per video-question analysis based on NSS
video_no, question_no = 2, 0

# get num_participants and num_frames for given video_no and question_no
num_participants = 0
num_frames = 0
for user_id, user_saliency_data in intra_all[(video_no, question_no)].items():
    num_participants += 1
    if len(user_saliency_data['per_frame_nsscores']) > num_frames:
        num_frames = len(user_saliency_data['per_frame_nsscores'])

# convert per-frame nss for all participants into a single numpy array
all_i_vs_all_but_i_per_frame_nss = np.zeros((num_participants, num_frames), dtype=np.float32)
for idx, (user_id, user_saliency_data) in enumerate(intra_all[(video_no, question_no)].items()):
    data_len = len(user_saliency_data['per_frame_nsscores'])
    all_i_vs_all_but_i_per_frame_nss[idx, :data_len] = user_saliency_data['per_frame_nsscores']
vq_nss_averaged_across_participants_per_frame = all_i_vs_all_but_i_per_frame_nss.mean(axis=0)

fig = plt.figure()
gs = fig.add_gridspec(num_participants + 1, hspace=0)
axs = gs.subplots(sharex=True)
for idx, ax in enumerate(axs):
    if idx == len(axs) - 1:
        ax.plot(vq_nss_averaged_across_participants_per_frame, color='r')
        ax.axhline(vq_nss_averaged_across_participants_per_frame.mean(), color='g')
    else:
        ax.plot(all_i_vs_all_but_i_per_frame_nss[idx])
plt.show()

In [ ]:
#%% Per video analysis based on NSS
video_no = 2

# get num_questions for given video_no
question_no_list = [k[1] for k in intra_all.keys() if k[0] == video_no]
# question_no_list = [0, 1, 2, 3, 4, 51, 6, 71]

# get vq_nss_averaged_across_participants_per_frame for each question in question_no_list
all_qs_nss_averaged_across_participants_per_frame = []
for question_no in question_no_list:
    # get num_participants and num_frames for given video_no and question_no
    num_participants = 0
    num_frames = 0
    for user_id, user_saliency_data in intra_all[(video_no, question_no)].items():
        num_participants += 1
        if len(user_saliency_data['per_frame_nsscores']) > num_frames:
            num_frames = len(user_saliency_data['per_frame_nsscores'])

    # convert per-frame nss for all participants into a single numpy array
    all_i_vs_all_but_i_per_frame_nss = np.zeros((num_participants, num_frames), dtype=np.float32)
    for idx, (user_id, user_saliency_data) in enumerate(intra_all[(video_no, question_no)].items()):
        data_len = len(user_saliency_data['per_frame_nsscores'])
        all_i_vs_all_but_i_per_frame_nss[idx, :data_len] = user_saliency_data['per_frame_nsscores']
    vq_nss_averaged_across_participants_per_frame = all_i_vs_all_but_i_per_frame_nss.mean(axis=0)
    all_qs_nss_averaged_across_participants_per_frame.append(vq_nss_averaged_across_participants_per_frame)
all_qs_nss_averaged_across_participants_per_frame = np.stack(all_qs_nss_averaged_across_participants_per_frame, axis=0)
nss_value_averaged_across_questions_per_frame = all_qs_nss_averaged_across_participants_per_frame.mean(axis=0)

# plot nss_averaged_across_participants_per_frame for all questions
fig = plt.figure()
gs = fig.add_gridspec(len(all_qs_nss_averaged_across_participants_per_frame), hspace=0)
axs = gs.subplots(sharex=True, sharey=True)
for idx, ax in enumerate(axs):
    ax.plot(all_qs_nss_averaged_across_participants_per_frame[idx] - nss_value_averaged_across_questions_per_frame, color='r')
plt.show()

### version 1 codes

In [ ]:
def get_questions_nss_averaged_across_participants_per_frame(intra_nss_values, video_no, question_no_list):
    # get num_questions for given video_no
    # question_no_list = [k[1] for k in intra_nss_values.keys() if k[0] == video_no]
    # question_no_list = [0, 1, 2, 3, 4, 51, 6, 71]
    if isinstance(question_no_list, int) and question_no_list == -1:
        question_no_list = [k[1] for k in intra_nss_values.keys() if k[0] == video_no]
    elif isinstance(question_no_list, list):
        question_no_list = question_no_list
    else:
        raise Exception("Unexpected value for question_no_list input argument! Expected either list of value or -1, but got {}".format(question_no_list))

    # get vq_nss_averaged_across_participants_per_frame for each question in question_no_list
    all_qs_nss_averaged_across_participants_per_frame = []
    for question_no in question_no_list:
        # get num_participants and num_frames for given video_no and question_no
        num_participants = 0
        num_frames = 0
        for user_id, user_saliency_data in intra_nss_values[(video_no, question_no)].items():
            num_participants += 1
            if len(user_saliency_data['per_frame_nsscores']) > num_frames:
                num_frames = len(user_saliency_data['per_frame_nsscores'])

        # convert per-frame nss for all participants into a single numpy array
        all_i_vs_all_but_i_per_frame_nss = np.zeros((num_participants, num_frames), dtype=np.float32)
        for idx, (user_id, user_saliency_data) in enumerate(intra_nss_values[(video_no, question_no)].items()):
            data_len = len(user_saliency_data['per_frame_nsscores'])
            all_i_vs_all_but_i_per_frame_nss[idx, :data_len] = user_saliency_data['per_frame_nsscores']
        vq_nss_averaged_across_participants_per_frame = all_i_vs_all_but_i_per_frame_nss.mean(axis=0)
        all_qs_nss_averaged_across_participants_per_frame.append(vq_nss_averaged_across_participants_per_frame)
        print(video_no, question_no, len(all_qs_nss_averaged_across_participants_per_frame[-1]))
    all_qs_nss_averaged_across_participants_per_frame = np.stack(all_qs_nss_averaged_across_participants_per_frame, axis=0)
    return all_qs_nss_averaged_across_participants_per_frame

def compute_correlations(averaged_nss):
    """
    Compute the Pearson correlation coefficient between each pair of rows in the 2D array 'averaged_nss',
    including each question with itself.

    Args:
    averaged_nss (numpy.ndarray): A 2D array of shape (num_questions, num_fixations), where each row contains 
                                  per-fixation/per-frame averages for a question.

    Returns:
    numpy.ndarray: A correlation matrix with dimensions (num_questions, num_questions).
    """
    num_questions = averaged_nss.shape[0]
    correlation_matrix = np.zeros((num_questions, num_questions))
    for i in range(num_questions):
        for j in range(i, num_questions):  # start j from i to include self-correlation
            correlation, _ = pearsonr(averaged_nss[i], averaged_nss[j])
            if i == j:
                correlation_matrix[i, j] = correlation
            else:
                correlation_matrix[i, j] = correlation
                correlation_matrix[j, i] = correlation  # Fill symmetric value
    return correlation_matrix

def plot_correlation_matrix(correlation_matrix, labels=None):
    """
    Plots a correlation matrix using seaborn heatmap function.

    Args:
    correlation_matrix (numpy.ndarray): A square array containing correlation coefficients.
    """
    if labels:
        labels = labels
    else:
        labels = [i for i in range(correlation_matrix.shape[0])]
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='PuOr',
                cbar=True, square=True, vmin=-1, vmax=1, xticklabels=labels, yticklabels=labels)
    plt.title('Correlation Matrix')
    plt.xlabel('Question Number')
    plt.ylabel('Question Number')
    plt.show()

In [ ]:
all_qs_nss_averaged_across_participants_per_frame = get_questions_nss_averaged_across_participants_per_frame(intra_nss_values=intra_all, video_no=6, question_no_list = -1)
nss_correlation_matrix = compute_correlations(all_qs_nss_averaged_across_participants_per_frame)
plot_correlation_matrix(nss_correlation_matrix)

In [ ]:
video_no_list = [i for i in range(0, 300)]
question_no_list = [0, 1, 2, 3, 4, 6]
for video_no in video_no_list:
    all_qs_nss_averaged_across_participants_per_frame = get_questions_nss_averaged_across_participants_per_frame(intra_nss_values=intra_all, video_no=video_no, question_no_list=question_no_list)
    if video_no == 0:
        nss_correlation_matrix = compute_correlations(all_qs_nss_averaged_across_participants_per_frame)
    else:
        nss_correlation_matrix += compute_correlations(all_qs_nss_averaged_across_participants_per_frame)
nss_correlation_matrix /= len(video_no_list)
plot_correlation_matrix(nss_correlation_matrix, question_no_list)

### version 2 codes (use these ones)

In [ ]:
def get_this_video_all_questions_nss_averaged_across_participants_per_frame(intra_nss_values, video_no, all_question_no_list):
    # get all available question_no's for given video_no
    this_video_question_nos = [k[1] for k in intra_nss_values.keys() if k[0] == video_no]

    # get vq_nss_averaged_across_participants_per_frame for each question in question_no_list
    all_questions_nss_averaged_across_participants_per_frame = {question_no: None for question_no in all_question_no_list}
    for question_no in this_video_question_nos:
        num_frames          = 900
        num_participants    = len(intra_nss_values[(video_no, question_no)].keys())
        # convert per-frame nss for all participants into a single numpy array
        all_i_vs_all_but_i_per_frame_nss = np.zeros((num_participants, num_frames), dtype=np.float32)
        for idx, (user_id, user_saliency_data) in enumerate(intra_nss_values[(video_no, question_no)].items()):
            data_len = len(user_saliency_data['per_frame_nsscores'])
            all_i_vs_all_but_i_per_frame_nss[idx, :data_len] = user_saliency_data['per_frame_nsscores']
        this_question_nss_averaged_across_participants_per_frame = all_i_vs_all_but_i_per_frame_nss.mean(axis=0)
        all_questions_nss_averaged_across_participants_per_frame[question_no] = this_question_nss_averaged_across_participants_per_frame
    return all_questions_nss_averaged_across_participants_per_frame


def compute_correlations(averaged_nss, all_question_no_list):
    """
    Compute the Pearson correlation coefficient between each pair of rows in the 2D array 'averaged_nss',
    including each question with itself.

    Args:
    averaged_nss (numpy.ndarray): A 2D array of shape (num_questions, num_fixations), where each row contains 
                                  per-fixation/per-frame averages for a question.

    Returns:
    numpy.ndarray: A correlation matrix with dimensions (num_questions, num_questions).
    """
    num_questions = len(all_question_no_list)
    correlation_matrix = np.full((num_questions, num_questions), np.nan, dtype=np.float32)
    for i in range(num_questions):
        i_q_no = all_question_no_list[i]
        if averaged_nss[i_q_no] is None:
            continue
        else:
            for j in range(i, num_questions):  # start j from i to include self-correlation
                j_q_no = all_question_no_list[j]
                if averaged_nss[j_q_no] is None:
                    continue
                else:
                    correlation, _ = pearsonr(averaged_nss[i_q_no], averaged_nss[j_q_no])
                    if i == j:
                        correlation_matrix[i, j] = correlation
                    else:
                        correlation_matrix[i, j] = correlation
                        correlation_matrix[j, i] = correlation  # Fill symmetric value
    return correlation_matrix

In [ ]:
video_no_list = [i for i in range(0, 300)]
all_question_no_list = [0, 1, 2, 3, 4, 6, 51, 52, 53, 54, 71, 72, 73, 74]
nss_correlation_matrix = []
for video_no in video_no_list:
    this_video_all_questions_nss_averaged_across_participants_per_frame = get_this_video_all_questions_nss_averaged_across_participants_per_frame(intra_nss_values=intra_all, video_no=video_no, all_question_no_list=all_question_no_list)
    nss_correlation_matrix.append(compute_correlations(this_video_all_questions_nss_averaged_across_participants_per_frame, all_question_no_list))
nss_correlation_matrix = np.stack(nss_correlation_matrix, axis=0)
# count each question type
cnt_questions = nss_correlation_matrix.shape[0] - np.isnan(nss_correlation_matrix).sum(axis=0)
# replace nan values with zeros
nss_correlation_matrix_no_nan = copy.deepcopy(nss_correlation_matrix)
nss_correlation_matrix_no_nan[np.isnan(nss_correlation_matrix)] = 0.0
# sum and average according to true count of each question type
nss_correlation_matrix_averaged_across_videos = nss_correlation_matrix_no_nan.sum(axis=0)/cnt_questions

In [ ]:
import matplotlib.colors as mcolors

def plot_correlation_matrix(correlation_matrix, labels=None):
    """
    Plots a correlation matrix using seaborn heatmap function.

    Args:
    correlation_matrix (numpy.ndarray): A square array containing correlation coefficients.
    """
    if labels:
        labels = labels
    else:
        labels = [i for i in range(correlation_matrix.shape[0])]
    plt.figure(figsize=(3.35, 3.35))
    fontsize = 5
    sns.set_theme(font_scale=1.8, font="Times New Roman")
    ax = sns.heatmap(correlation_matrix, annot=False, fmt="0.2f", cmap='PuOr', annot_kws={'size': 6}, #linewidths=0, linecolor='green',
                     cbar=True, cbar_kws={'shrink': .7}, square=True, vmin=-1, vmax=1, xticklabels=labels, yticklabels=labels)

    # Create a reversed grayscale colormap for the range [0, 0.5]
    norm = mcolors.Normalize(vmin=0.0, vmax=1.0)
    cmap = plt.cm.get_cmap("Greys_r")  # Reversed grayscale colormap (white to black)
    
    # Custom annotations
    for i in range(correlation_matrix.shape[0]):
        for j in range(correlation_matrix.shape[1]):
            value = correlation_matrix[i, j]
            if i == j:  # Diagonal elements
                annotation = f"{value:.1f}"
                color = "white"  # White font for diagonal elements
            else:  # Off-diagonal elements
                annotation = f"{value:.2f}".lstrip('0')  # Remove leading zero
                # Map the value to a reversed grayscale color for [0, 0.5]
                color = cmap(float(value > 0.4))
            ax.text(j + 0.5, i + 0.5, annotation, ha='center', va='center', fontsize=5, color=color)
    

    colorbar = ax.collections[0].colorbar
    colorbar.ax.tick_params(labelsize=5, length=0)  # Adjust the font size here
    ticks = np.linspace(-1, 1, 5)  # Generate evenly spaced ticks between -1 and 1
    colorbar.set_ticks(ticks)
    plt.title('Pearson Correlation Matrix', fontsize=fontsize, pad=5)
    plt.xlabel('Question Type No', fontsize=fontsize, labelpad=1)
    plt.ylabel('Question Type No', fontsize=fontsize, labelpad=1)
    # Rotate y-axis labels and remove the y-axis tick marks
    plt.yticks(rotation=0, fontsize=5)
    plt.xticks(rotation=0, fontsize=5)
    plt.tick_params(axis='both', which='both', length=1, width=0)
    
    plt.savefig('./output/r3_analysis_results/'+'Pearson_Correlation_inter_question_resized.png', dpi=300, bbox_inches='tight')
    plt.show()

# plot confusion matrix
plot_correlation_matrix(nss_correlation_matrix_averaged_across_videos, all_question_no_list)


In [ ]:
def plot_correlation_matrix(correlation_matrix, labels=None):
    """
    Plots a correlation matrix using seaborn heatmap function.

    Args:
    correlation_matrix (numpy.ndarray): A square array containing correlation coefficients.
    """
    if labels:
        labels = labels
    else:
        labels = [i for i in range(correlation_matrix.shape[0])]
    plt.figure(figsize=(6.6, 6.6))
    fontsize = 9
    sns.set_theme(font_scale=1.8, font="Times New Roman")
    ax = sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='PuOr', annot_kws={'size': 8}, #linewidths=0, linecolor='green',
                     cbar=True, cbar_kws={'shrink': .8}, square=True, vmin=-1, vmax=1, xticklabels=labels, yticklabels=labels)
    

    colorbar = ax.collections[0].colorbar
    colorbar.ax.tick_params(labelsize=fontsize, length=0)  # Adjust the font size here
    ticks = np.linspace(-1, 1, 5)  # Generate evenly spaced ticks between -1 and 1
    colorbar.set_ticks(ticks)
    plt.title('Pearson Correlation Matrix', fontsize=fontsize, pad=5)
    plt.xlabel('Question Type No', fontsize=fontsize, labelpad=5)
    plt.ylabel('Question Type No', fontsize=fontsize, labelpad=5)
    # Rotate y-axis labels and remove the y-axis tick marks
    plt.yticks(rotation=0, fontsize=fontsize)
    plt.xticks(rotation=0, fontsize=fontsize)
    plt.tick_params(axis='both', which='both', length=3, width=0)
    
    plt.savefig('./output/r3_analysis_results/'+'Pearson_Correlation_inter_question_highres.png', dpi=300, bbox_inches='tight')
    plt.show()    

# plot confusion matrix
plot_correlation_matrix(nss_correlation_matrix_averaged_across_videos, all_question_no_list)

### detailed visualizations for paper

In [ ]:
video_no = 51
question_no = 54

num_frames = 900
num_participants = len(intra_all[(video_no, question_no)].keys())
# convert per-frame nss for all participants into a single numpy array
user_ids_list = []
all_i_vs_all_but_i_per_frame_nss = np.zeros((num_participants, num_frames), dtype=np.float32)
for idx, (user_id, user_saliency_data) in enumerate(intra_all[(video_no, question_no)].items()):
    data_len = len(user_saliency_data['per_frame_nsscores'])
    all_i_vs_all_but_i_per_frame_nss[idx, :data_len] = user_saliency_data['per_frame_nsscores']
    user_ids_list.append(user_id)
this_question_nss_averaged_across_participants_per_frame = all_i_vs_all_but_i_per_frame_nss.mean(axis=0)

In [ ]:
print("min nss val:", all_i_vs_all_but_i_per_frame_nss.min())
print("max nss val:", all_i_vs_all_but_i_per_frame_nss.max())
print("min nss val:", this_question_nss_averaged_across_participants_per_frame.min())
print("max nss val:", this_question_nss_averaged_across_participants_per_frame.max())

In [ ]:
for current_idx in range(0, len(all_i_vs_all_but_i_per_frame_nss)):
    current_user_id = user_ids_list[current_idx]
    plt.rcParams["font.family"] = "Times New Roman"
    plt.figure(figsize=(6.6, 1.2))
    fontsize = 9
    plt.xlim(0, num_frames)
    plt.ylim(-1.4, 11.4)
    plt.xticks([0, 100, 200, 300, 400, 500, 600, 700, 800, 900], fontsize=fontsize)
    plt.yticks([-1, 5, 11], fontsize=fontsize)
    plt.tick_params(axis='both', direction='in', which='both', length=5, width=0)
    plt.grid(True, axis='both', which='major', linestyle='--', alpha=0.8, color='magenta')
    plt.plot(all_i_vs_all_but_i_per_frame_nss[current_idx], color='b', linewidth=2)
    nss_averaged_across_frames = sum(all_i_vs_all_but_i_per_frame_nss[current_idx])/len(all_i_vs_all_but_i_per_frame_nss[current_idx])
    plt.savefig('./output/r3_analysis_results/visualizations_for_paper/'+'nss_values_v_{}_q_{}_p_{}_nss_{:.2f}.png'.format(video_no, question_no, current_user_id, nss_averaged_across_frames), dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
plt.rcParams["font.family"] = "Times New Roman"
plt.figure(figsize=(6.6, 1.2))
fontsize = 9
plt.xlim(0, num_frames)
plt.ylim(-1.4, 11.4)
plt.xticks([0, 100, 200, 300, 400, 500, 600, 700, 800, 900], fontsize=fontsize)
plt.yticks([-1, 5, 11], fontsize=fontsize)
plt.tick_params(axis='both', direction='in', which='both', length=5, width=0)
plt.grid(True, axis='both', which='major', linestyle='--', alpha=0.8, color='magenta')
plt.plot(this_question_nss_averaged_across_participants_per_frame, color='r', linewidth=2)
nss_averaged_across_frames = this_question_nss_averaged_across_participants_per_frame.mean()
plt.axhline(nss_averaged_across_frames, linestyle='--', linewidth=2, alpha=0.6, color='crimson')
# plt.text(430, 1.7, "mean", fontsize=fontsize, alpha=0.6, color='crimson')
plt.savefig('./output/r3_analysis_results/visualizations_for_paper/'+'t-nss_v_{}_q_{}_mean_{:.2f}.png'.format(video_no, question_no, nss_averaged_across_frames), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
questions_nss_averaged_across_participants_per_frame = {}

In [ ]:
questions_nss_averaged_across_participants_per_frame[(video_no, question_no)] = copy.deepcopy(this_question_nss_averaged_across_participants_per_frame)

In [ ]:
for vq_i in questions_nss_averaged_across_participants_per_frame.keys():
    for vq_j in questions_nss_averaged_across_participants_per_frame.keys():
        correlation, _ = pearsonr(questions_nss_averaged_across_participants_per_frame[vq_i], questions_nss_averaged_across_participants_per_frame[vq_j])
        print("Pearson correlation between {} and {} video-question pairs is: {:.3f}".format(vq_i, vq_j, correlation))

### testing

In [ ]:
# def get_this_video_this_question_all_participants_nss_per_frame(intra_nss_values, video_no, all_question_no_list):
#     # get all available question_no's for given video_no
#     this_video_question_nos = [k[1] for k in intra_nss_values.keys() if k[0] == video_no]

#     # get vq_nss_averaged_across_participants_per_frame for each question in question_no_list
#     all_questions_nss_averaged_across_participants_per_frame = {question_no: None for question_no in all_question_no_list}
#     for question_no in this_video_question_nos:
#         num_frames          = 900
#         num_participants    = len(intra_nss_values[(video_no, question_no)].keys())
#         # convert per-frame nss for all participants into a single numpy array
#         all_i_vs_all_but_i_per_frame_nss = np.zeros((num_participants, num_frames), dtype=np.float32)
#         for idx, (user_id, user_saliency_data) in enumerate(intra_nss_values[(video_no, question_no)].items()):
#             data_len = len(user_saliency_data['per_frame_nsscores'])
#             all_i_vs_all_but_i_per_frame_nss[idx, :data_len] = user_saliency_data['per_frame_nsscores']
#         this_question_nss_averaged_across_participants_per_frame = all_i_vs_all_but_i_per_frame_nss.mean(axis=0)
#         all_questions_nss_averaged_across_participants_per_frame[question_no] = this_question_nss_averaged_across_participants_per_frame
#     return all_questions_nss_averaged_across_participants_per_frame

bins_test = np.linspace(0, 1, 20)

question_correlations = []
for video_no in range(0, 300):
    question_no = 51
    if (video_no, question_no) in intra_all.keys(): 
        num_frames          = 900
        num_participants    = len(intra_all[(video_no, question_no)].keys())

        # convert per-frame nss for all participants into a single numpy array
        all_i_vs_all_but_i_per_frame_nss = np.zeros((num_participants, num_frames), dtype=np.float32)
        for idx, (user_id, user_saliency_data) in enumerate(intra_all[(video_no, question_no)].items()):
            data_len = len(user_saliency_data['per_frame_nsscores'])
            all_i_vs_all_but_i_per_frame_nss[idx, :data_len] = user_saliency_data['per_frame_nsscores']

        correlation_avg = 0
        for idx, user_id in enumerate(intra_all[(video_no, question_no)].keys()):
            correlation, _ = pearsonr(all_i_vs_all_but_i_per_frame_nss[idx], np.concatenate([all_i_vs_all_but_i_per_frame_nss[:idx], all_i_vs_all_but_i_per_frame_nss[idx+1:]]).mean(axis=0)) #all_i_vs_all_but_i_per_frame_nss.mean(0))
            correlation_avg += correlation
            question_correlations.append(correlation)
        correlation_avg /= len(intra_all[(video_no, question_no)].keys())

        # question_correlations.append(correlation_avg)

plt.hist(x=question_correlations, bins=bins_test)
mean = sum(question_correlations)/len(question_correlations)
plt.axvline(mean, color='r', label='mean={:.2f}'.format(mean))
plt.legend()
